# Module X1: Repeated measures and mixed-effects models

**Goal:** analyse data where the **same units are measured several times** (or where rows are **grouped** in clusters: students in classes, patients in hospitals). Ordinary tests and regression assume independent rows; here they aren't.
Ignoring that is one of the most common reasons a thesis analysis is rejected.

**You will learn:** why clustered data break independence · the intraclass correlation (ICC) and the design effect · the cost of ignoring clustering (a simulation) ·
the simple **summary-measures** approach · **linear mixed-effects models** (random intercepts and slopes) · cluster-robust standard errors · choosing a random-effects structure · unbalanced data · reporting

**Maths in this module:** variance components · ICC $=\sigma_u^2/(\sigma_u^2+\sigma_\varepsilon^2)$ · design effect $1+(m-1)\,\text{ICC}$ · the mixed model $y_{ij}=\mathbf x_{ij}^\top\beta+u_{0i}+u_{1i}t_{ij}+\varepsilon_{ij}$ · likelihood-based model comparison

> 🔗 **Prerequisites:** Modules 8b (tests, paired designs), 9 (regression) and 7b (likelihood, AIC). Read the paired $t$-test section of 8b again if needed: a paired design is the simplest repeated-measures design.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
long = pd.read_csv(DATA / "longitudinal_scores.csv")
long.head(8)

## The data

`longitudinal_scores.csv`: **80 students**, half in a *support programme*, each scored at up to **5 waves** (0–4). About 5% of rows are missing (some students skipped a late wave), so the data are **unbalanced**.
**Question:** does the programme improve scores *faster* than the control condition? (an effect on the **rate of change**, i.e. a group × time interaction).

In [ ]:
print(long.shape, "| students:", long["student_id"].nunique(), "| rows per wave:", long["wave"].value_counts().sort_index().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, (grp, d) in zip(axes, long.groupby("group")):
    for sid, s in d.groupby("student_id"):
        ax.plot(s["wave"], s["score"], color="grey", alpha=0.35, linewidth=0.8)
    ax.plot(sorted(d["wave"].unique()), d.groupby("wave")["score"].mean(), color="tab:red", linewidth=3, label="group mean")
    ax.set_title(f"{grp}: one line per student"); ax.set_xlabel("wave"); ax.legend()
axes[0].set_ylabel("Score")
plt.tight_layout(); plt.show()

The grey "spaghetti" lines show it: students **start at very different levels** and mostly stay near their own line. A student's five scores are far more alike than scores of different students: **rows are not independent**.

## 1. How dependent? The intraclass correlation (ICC)

Split the variation in scores into **between-student** variance $\sigma_u^2$ and **within-student** (residual) variance $\sigma_\varepsilon^2$. The **ICC** is the share that is between students:

$$\text{ICC}=\frac{\sigma_u^2}{\sigma_u^2+\sigma_\varepsilon^2}$$

It's also the correlation between two measurements on the same student. For a *balanced* design ($n$ students with $k$ measurements each), the ANOVA estimator is
$\widehat{\text{ICC}}=\dfrac{\text{MSB}-\text{MSW}}{\text{MSB}+(k-1)\,\text{MSW}}$ (with MSB, MSW the between- and within-student mean squares from a one-way ANOVA with student as the factor).
Waves 0–2 are complete for all 80 students, so use them:

In [ ]:
def icc_balanced(df, subject, value):
    groups = [g[value].to_numpy() for _, g in df.groupby(subject)]
    n, k = len(groups), len(groups[0])
    assert all(len(g) == k for g in groups), "needs a balanced design"
    grand = np.concatenate(groups).mean()
    msb = k * sum((g.mean() - grand) ** 2 for g in groups) / (n - 1)
    msw = sum(((g - g.mean()) ** 2).sum() for g in groups) / (n * (k - 1))
    return (msb - msw) / (msb + (k - 1) * msw), msb, msw

icc, msb, msw = icc_balanced(long[long["wave"] <= 2], "student_id", "score")
print(f"MSB = {msb:.1f}, MSW = {msw:.1f}  ->  ICC = {icc:.3f}")

m = 5                                                  # measurements per student in the full design
deff = 1 + (m - 1) * icc
print(f"design effect = 1 + (m-1)*ICC = {deff:.2f}  -> 400 rows carry about as much information as {400 / deff:.0f} independent ones")

An ICC of about **0.77** means most of the variation is between students. The **design effect** $1+(m-1)\text{ICC}$ says how many times larger the variance of an average is than if all rows were independent;
dividing the nominal sample size by it gives the **effective sample size**.

## 2. What goes wrong if you ignore it?

Simulate a world with **no programme effect at all**, but with student-level differences (ICC = 0.8), then test the group difference two ways: treating the 5 rows per student as independent (naive OLS), and with **cluster-robust** standard errors.
If a method is valid, it should reject a true null about 5% of the time.

In [ ]:
rng = np.random.default_rng(0)
n_sims, reject_naive, reject_cluster = 600, 0, 0
for _ in range(n_sims):
    u = rng.normal(0, 8, 40)                                           # 40 students, between-student sd 8
    d = pd.DataFrame({"y": (u[:, None] + rng.normal(0, 4, (40, 5))).ravel(),   # within-student sd 4 -> ICC = 0.8
                      "g": np.repeat(np.r_[np.zeros(20), np.ones(20)], 5),
                      "s": np.repeat(np.arange(40), 5)})
    reject_naive += smf.ols("y ~ g", d).fit().pvalues["g"] < 0.05
    reject_cluster += smf.ols("y ~ g", d).fit(cov_type="cluster", cov_kwds={"groups": d["s"]}).pvalues["g"] < 0.05
print(f"false-positive rate, naive OLS (rows treated as independent): {reject_naive / n_sims:.1%}")
print(f"false-positive rate, cluster-robust standard errors         : {reject_cluster / n_sims:.1%}")

Ignoring clustering made a nominal 5% test reject a true null about **one time in three**. That is because treating 200 correlated rows per group as independent *pretends* we have about four times more information than we do.
(The direction of the error depends on the effect: for **between-student** effects like a group main effect, naive standard errors are too *small*; for **within-student** effects like change over time, they can be too *large*, since the large between-student variance isn't removed.)

## 3. The simplest valid approach: summary measures

Reduce each student to **one number that answers your question**, then compare those numbers between groups with an ordinary test. For "who improves faster?", the summary is each student's **own slope** of score on wave. Students are independent of one another, so the usual $t$-test is valid.

In [ ]:
slopes = (long.groupby(["student_id", "group"])
              .apply(lambda d: np.polyfit(d["wave"], d["score"], 1)[0], include_groups=False)
              .reset_index(name="slope"))
prog, ctrl = slopes.loc[slopes["group"] == "programme", "slope"], slopes.loc[slopes["group"] == "control", "slope"]
res = stats.ttest_ind(prog, ctrl, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"mean slope: programme {prog.mean():.2f} vs control {ctrl.mean():.2f} points per wave")
print(f"difference {prog.mean() - ctrl.mean():.2f} (95% CI {ci.low:.2f} to {ci.high:.2f}), Welch t = {res.statistic:.2f}, p = {res.pvalue:.1e}  (n = {len(prog)} and {len(ctrl)} students)")

The programme students improve about **1.9 points per wave faster** (95% CI 1.1 to 2.7). The summary-measures approach is transparent, easy to explain and robust. Its weaknesses: it ignores that some students' slopes are estimated from fewer waves (so it weights everyone equally),
can't include time-varying covariates easily, and doesn't use all the information. A **mixed model** fixes those.

## 4. Linear mixed-effects models

A mixed model has **fixed effects** (the population-level effects you care about) and **random effects** (student-specific deviations, treated as draws from a distribution):

$$y_{ij}=\underbrace{\beta_0+\beta_1\,\text{wave}_{ij}+\beta_2\,\text{prog}_i+\beta_3\,\text{wave}_{ij}\,\text{prog}_i}_{\text{fixed}}+\underbrace{u_{0i}+u_{1i}\,\text{wave}_{ij}}_{\text{random}}+\varepsilon_{ij}$$

where $u_{0i}\sim N(0,\sigma_0^2)$ is student $i$'s **random intercept** (own starting level), $u_{1i}\sim N(0,\sigma_1^2)$ their **random slope** (own growth rate), and $\varepsilon_{ij}\sim N(0,\sigma^2)$.
**$\beta_3$ is our target**: the extra growth per wave for programme students.

In [ ]:
model = smf.mixedlm("score ~ wave * group", data=long, groups=long["student_id"], re_formula="~wave").fit(reml=True)
print(model.summary().tables[1])

In [ ]:
fixed = pd.DataFrame({"estimate": model.fe_params, "ci_low": model.conf_int().loc[model.fe_params.index, 0],
                      "ci_high": model.conf_int().loc[model.fe_params.index, 1]}).round(2)
print(fixed)
s2_int, s2_slope, s2_res = model.cov_re.iloc[0, 0], model.cov_re.iloc[1, 1], model.scale
print(f"\nvariance components: intercept sd = {math.sqrt(s2_int):.1f}, slope sd = {math.sqrt(s2_slope):.2f}, residual sd = {math.sqrt(s2_res):.1f}")

**Reading the output**

* `wave` (≈ 1.75): control students gain about 1.75 points per wave.
* **`wave:group[T.programme]` (≈ 1.86; 95% CI 1.08 to 2.64):** programme students gain about 1.9 *more* points per wave. This agrees with the summary-measures answer (1.87).
* `group[T.programme]` (≈ 1.7, CI −2.2 to 5.5): the groups' *starting* levels (at wave 0) don't differ significantly, as expected with random assignment.
* **Variance components:** the sd of starting levels is about 8.3 points; students' growth rates vary with an sd of about 1.3 points per wave around the average; the leftover noise within a student's line is about 3.6 points.

### How different are the standard errors?

In [ ]:
ols = smf.ols("score ~ wave * group", long).fit()
cluster = smf.ols("score ~ wave * group", long).fit(cov_type="cluster", cov_kwds={"groups": long["student_id"]})
term_inter, term_group = "wave:group[T.programme]", "group[T.programme]"
table = pd.DataFrame({
    "naive OLS": [ols.bse[term_inter], ols.bse[term_group]],
    "cluster-robust OLS": [cluster.bse[term_inter], cluster.bse[term_group]],
    "mixed model": [model.bse[term_inter], model.bse[term_group]],
}, index=["SE of interaction (within-student effect)", "SE of group effect (between-student effect)"]).round(3)
table

Exactly the pattern described above. For the **interaction** (a within-student effect) the naive SE (0.69) is too *large*; for the **group** effect (a between-student effect) it is too *small* (1.64 vs 1.96). Cluster-robust and mixed-model standard errors agree for the interaction (0.41 vs 0.40).

## 5. Choosing the random-effects structure

Do students differ only in *starting level* (random intercept), or also in *growth rate* (random slope)? Compare the two fits by AIC. When the models differ in their **random effects**, fit by REML for the final model, but compare **fixed-effect** structures with maximum likelihood (`reml=False`).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ml_int = smf.mixedlm("score ~ wave * group", long, groups=long["student_id"]).fit(reml=False)
    ml_slope = smf.mixedlm("score ~ wave * group", long, groups=long["student_id"], re_formula="~wave").fit(reml=False)

def aic(res, k_cov):                                    # fixed-effect parameters + variance/covariance parameters + residual
    return -2 * res.llf + 2 * (len(res.fe_params) + k_cov + 1)

print(f"random intercept only : AIC = {aic(ml_int, 1):.1f}")
print(f"random intercept+slope: AIC = {aic(ml_slope, 3):.1f}   (lower is better)")

Allowing growth rates to differ by student lowers AIC by about 17: the random-slope model is clearly better. If you ignored this, your standard error for the interaction would be too small, because the model would assume everyone grows at the same rate.
**Rule:** include random slopes for effects that **vary by cluster and that you test**; start with the "maximal" structure your data can support, and simplify if the model won't converge (see the pitfalls in the README).

## 6. Checking the model

In [ ]:
fitted = model.fittedvalues
resid = model.resid
random_intercepts = pd.Series({sid: eff.iloc[0] for sid, eff in model.random_effects.items()})

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].scatter(fitted, resid, s=10, alpha=0.6); axes[0].axhline(0, color="tab:red")
axes[0].set_xlabel("fitted"); axes[0].set_ylabel("residual"); axes[0].set_title("residuals vs fitted: shapeless is good")
stats.probplot(resid, plot=axes[1]); axes[1].set_title("Q-Q: residuals")
stats.probplot(random_intercepts, plot=axes[2]); axes[2].set_title("Q-Q: student random intercepts")
plt.tight_layout(); plt.show()
print(f"Shapiro-Wilk on random intercepts: p = {stats.shapiro(random_intercepts).pvalue:.2f}")

## 7. Unbalanced data and missing waves

Here 4.75% of rows are missing and only 61 of the 80 students have all five waves. Repeated-measures ANOVA would have to **drop every student with a missing wave**. A mixed model uses **all available rows**, and is valid when data are
**missing at random** (missingness may depend on things you observed, like earlier scores, but not on the unobserved value itself).

In [ ]:
per_student = long.groupby("student_id").size()
complete_ids = per_student[per_student == 5].index
print(f"rows: {len(long)} of {80 * 5} possible ({1 - len(long) / 400:.2%} missing); students with all 5 waves: {len(complete_ids)} of 80")

for label, data in {"all available rows": long, "complete cases only": long[long["student_id"].isin(complete_ids)]}.items():
    fit = smf.mixedlm("score ~ wave * group", data, groups=data["student_id"]).fit()            # random intercept (stable on small subsets)
    est, (lo, hi) = fit.params[term_inter], fit.conf_int().loc[term_inter]
    print(f"{label:20s} n students = {data['student_id'].nunique():2d}: interaction {est:.2f} (95% CI {lo:.2f} to {hi:.2f})")

Using every row gives a narrower interval, because it uses more information. (If missingness could depend on the *unobserved* score, e.g. weak students drop out, none of these analyses is safe; run a **sensitivity analysis** and discuss it.)

## 8. A figure for the thesis: model trajectories with uncertainty

In [ ]:
waves = np.arange(5)
grid = pd.DataFrame({"wave": np.tile(waves, 2), "group": np.repeat(["control", "programme"], 5)})
X = np.column_stack([np.ones(len(grid)), grid["wave"], (grid["group"] == "programme").astype(float), grid["wave"] * (grid["group"] == "programme")])
fe_names = ["Intercept", "wave", "group[T.programme]", "wave:group[T.programme]"]
beta = model.fe_params[fe_names].to_numpy()
cov = model.cov_params().loc[fe_names, fe_names].to_numpy()
pred = X @ beta
se = np.sqrt(np.einsum("ij,jk,ik->i", X, cov, X))                       # SE of each predicted mean from the fixed-effect covariance

fig, ax = plt.subplots(figsize=(6.5, 4))
for g, color in (("control", "tab:blue"), ("programme", "tab:orange")):
    idx = (grid["group"] == g).to_numpy()
    ax.plot(grid.loc[idx, "wave"], pred[idx], color=color, linewidth=2, label=g)
    ax.fill_between(grid.loc[idx, "wave"], pred[idx] - 1.96 * se[idx], pred[idx] + 1.96 * se[idx], color=color, alpha=0.2)
ax.set_xlabel("Wave"); ax.set_ylabel("Predicted mean score"); ax.set_title("Programme students improve faster")
ax.legend(); plt.show()

## 9. Which method for which design?

| design | simple option | model-based option |
|---|---|---|
| 2 time points, same people | paired $t$ / Wilcoxon (Module 8b) | mixed model (random intercept) |
| 3+ time points, balanced, no missing | repeated-measures ANOVA (check sphericity) | mixed model |
| trend over time, group comparison | **summary measures** (slope per person) | **mixed model with random slopes** |
| unbalanced / missing waves / time-varying covariates | summary measures with care | **mixed model** |
| clustered rows (students in classes) | analyse cluster means; cluster-robust SEs | **mixed model with a random intercept per cluster** |
| binary or count outcomes, repeated | – | GEE or generalised mixed models (logistic/Poisson) |

### Reporting template
> *Scores were modelled with a linear mixed-effects model with fixed effects for wave, group and their interaction, and random intercepts and slopes for students (REML; `statsmodels`). Programme students gained 1.86 more points per wave than control students (95% CI 1.08 to 2.64, p < .001; 80 students, 381 observations). The ICC of 0.77 indicated strong clustering, so cluster-aware methods were required; a summary-slopes analysis agreed (difference 1.87, 95% CI 1.06 to 2.68).
> Data were missing for 4.75% of observations; the model assumes data are missing at random.*

## Recap

* Repeated or clustered rows are **not independent**. Measure the dependence with the **ICC** and the **design effect**.
* Ignoring it can inflate false positives enormously (here ≈ 34% instead of 5%) or shrink power, depending on whether the effect is between or within clusters.
* **Summary measures** are simple and valid; **mixed models** use all the data and handle unbalanced designs; **cluster-robust SEs** are a quick safeguard.
* Choose random effects by the structure of the data (and compare by AIC); check residuals and random effects; report assumptions about missing data.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** in the README.